## 0) **Dataset Auto‑Download (Optional)**

Run this cell to download and arrange Flickr8k and/or COCO 2017.

In [ ]:
import urllib.request, zipfile, shutil, os, sys
from pathlib import Path

def _progress_hook(block_num, block_size, total_size):
    downloaded = block_num * block_size
    total_size = max(total_size, 1)
    percent = min(100.0, downloaded / total_size * 100)
    sys.stdout.write(f'  -> {percent:5.1f}% ({downloaded/1e6:,.1f} MB / {total_size/1e6:,.1f} MB)')
    sys.stdout.flush()

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)

def download_file(url: str, dest: Path):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        print(f'Already exists: {dest}')
        return dest
    print(f'Downloading\n  URL : {url}\n  --> : {dest}')
    urllib.request.urlretrieve(url, dest, _progress_hook)
    print('\nDone.')
    return dest

def extract_zip(zip_path: Path, target_dir: Path):
    if target_dir.exists() and any(target_dir.iterdir()):
        print(f'Already extracted: {target_dir}')
        return
    print(f'Extracting {zip_path} -> {target_dir}')
    target_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(target_dir)
    print('Done.')

FLICKR8K_URLS = {
    'images_zip': 'https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_Dataset.zip',
    'text_zip'  : 'https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_text.zip',
}

def prepare_flickr8k(cfg):
    root = Path(cfg['data_root'])/ 'Flickr8k'
    img_dir = root / 'images'
    cap_dir = root / 'captions'
    ensure_dir(img_dir); ensure_dir(cap_dir)
    images_zip = download_file(FLICKR8K_URLS['images_zip'], root/'Flickr8k_Dataset.zip')
    text_zip   = download_file(FLICKR8K_URLS['text_zip'],   root/'Flickr8k_text.zip')
    extract_zip(images_zip, root/'_extract_images')
    extract_zip(text_zip,   root/'_extract_text')
    flicker_dir = None
    for p in (root/'_extract_images').glob('*'):
        if p.is_dir(): flicker_dir = p; break
    if flicker_dir is None:
        print('ERROR: Could not find extracted image folder.')
    else:
        jpgs = list(flicker_dir.glob('*.jpg'))
        if jpgs and not any(img_dir.iterdir()):
            print(f'Moving {len(jpgs)} images -> {img_dir}')
            for p in jpgs:
                tgt = img_dir / p.name
                if not tgt.exists(): shutil.move(str(p), str(tgt))
        else:
            print('Images already present or none found to move.')
    text_dir = None
    for p in (root/'_extract_text').glob('*'):
        if p.is_dir(): text_dir = p; break
    if text_dir is None:
        print('ERROR: Could not find extracted text folder.')
    else:
        needed = ['Flickr8k.token.txt','Flickr_8k.trainImages.txt','Flickr_8k.devImages.txt','Flickr_8k.testImages.txt']
        for name in needed:
            src = text_dir / name; dst = cap_dir / name
            if not dst.exists(): shutil.copy2(src, dst)
        print(f'Caption files placed in: {cap_dir}')
    for p in [root/'_extract_images', root/'_extract_text']:
        try: shutil.rmtree(p)
        except Exception: pass
    print('Flickr8k prepared at:', root)

COCO_URLS = {
    'train_images': 'http://images.cocodataset.org/zips/train2017.zip',
    'val_images'  : 'http://images.cocodataset.org/zips/val2017.zip',
    'annotations' : 'http://images.cocodataset.org/annotations/annotations_trainval2017.zip',
}

def prepare_coco2017(cfg):
    root = Path(cfg['data_root'])/ 'COCO'
    train_dir = root/'train2017'
    val_dir   = root/'val2017'
    ann_dir   = root/'annotations'
    ensure_dir(train_dir); ensure_dir(val_dir); ensure_dir(ann_dir)
    train_zip = download_file(COCO_URLS['train_images'], root/'train2017.zip')
    val_zip   = download_file(COCO_URLS['val_images'],   root/'val2017.zip')
    ann_zip   = download_file(COCO_URLS['annotations'],  root/'annotations_trainval2017.zip')
    if not any(train_dir.iterdir()): extract_zip(train_zip, root)
    else: print('train2017 already extracted.')
    if not any(val_dir.iterdir()): extract_zip(val_zip, root)
    else: print('val2017 already extracted.')
    if not any(ann_dir.iterdir()): extract_zip(ann_zip, root)
    else: print('annotations already extracted.')
    print('COCO prepared at:', root)

def auto_prepare_datasets(cfg, do_flickr=True, do_coco=False):
    if do_flickr:
        print('=== Preparing Flickr8k ==='); prepare_flickr8k(cfg)
    if do_coco:
        print('=== Preparing COCO 2017 ==='); prepare_coco2017(cfg)
    print('All requested datasets are ready.')

In [ ]:
# Example:
# auto_prepare_datasets(CFG, do_flickr=True, do_coco=False)
# auto_prepare_datasets(CFG, do_flickr=False, do_coco=True)